# County population by census year (IPUMS NHGIS API)

This notebook shows how to request **nominal** county-level **total population** from IPUMS NHGIS using the official Python client (`ipumspy`), load the result into pandas, save a wide CSV (one row per county, census years as columns), and plot a simple time series.

## Prerequisites

1. An [IPUMS account](https://www.nhgis.org/) with **NHGIS** registration.
2. An API key from [account.ipums.org/api_keys](https://account.ipums.org/api_keys).
3. Set `IPUMS_API_KEY` in your environment, **or** create a `.env` file in this folder (see below). The `.env` file is listed in `.gitignore` so it is not committed.

## Data choice

NHGIS **time series table [A00](https://www.nhgis.org/time-series-tables)** (*Total Population*, **nominal** geographic integration) links comparable total-population counts across decennial censuses at the **state–county** level. Each row is an NHGIS county unit; a given census column is non-empty when that unit appears in that census (per NHGIS nominal linkage). To approximate **current** counties, we keep counties with non-missing **2020** population (you can change this rule if you prefer another baseline).

References: [IPUMS API overview](https://developer.ipums.org/docs/v2/apiprogram/), [NHGIS](https://www.nhgis.org/), [ipumspy aggregate extracts](https://ipumspy.readthedocs.io/en/latest/ipums_api/ipums_api_aggregate/index.html).

In [ ]:
import os
import re
import zipfile
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from ipumspy import (
    AggregateDataExtract,
    IpumsApiClient,
    TimeSeriesTable,
)
from ipumspy.api.metadata import TimeSeriesTableMetadata

# Output directory (this folder)
OUT_DIR = Path(".")


def _load_env_file(path: Path) -> None:
    """Load KEY=value lines into os.environ if not already set."""
    if not path.is_file():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, val = line.partition("=")
        key, val = key.strip(), val.strip().strip('"').strip("'")
        if key:
            os.environ.setdefault(key, val)


_load_env_file(OUT_DIR / ".env")

DATA_DIR = OUT_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

ZIP_PATH = DATA_DIR / "nhgis_county_a00_extract.zip"
CSV_OUT = DATA_DIR / "county_population_by_census_year.csv"

## Optional: inspect metadata for table A00

Confirms available geographic levels and census years before submitting an extract.

In [ ]:
api_key = os.environ.get("IPUMS_API_KEY")
if not api_key:
    raise RuntimeError(
        "Set IPUMS_API_KEY in your environment or add it to .env in this folder "
        "(see https://account.ipums.org/api_keys)."
    )

client = IpumsApiClient(api_key)
meta = client.get_metadata(TimeSeriesTableMetadata("A00"))
print("Description:", meta.description)
print("Geographic integration:", getattr(meta, "geographic_integration", ""))
print("Geographic levels:", meta.geog_levels)
print("Years (sample):", (getattr(meta, "years", None) or [])[:5], "...")

## Submit extract, wait, and download

- **A00** at **county**: total population, nominal integration, years 1790–2020 (all available years by default).
- `tst_layout="time_by_column_layout"`: time points as separate columns (default).
- `data_format="csv_header"`: include the descriptive second header row (we skip it when reading into pandas).

In [ ]:
extract = AggregateDataExtract(
    collection="nhgis",
    description="County total population (A00 nominal), all census years",
    time_series_tables=[
        TimeSeriesTable("A00", geog_levels=["county"]),
    ],
    data_format="csv_header",
    tst_layout="time_by_column_layout",
)

submitted = client.submit_extract(extract)
client.wait_for_extract(submitted)
client.download_extract(submitted, str(ZIP_PATH))
print("Saved:", ZIP_PATH)

## Load CSV from the zip

NHGIS zips contain a `*_csv/` folder with one or more `.csv` files. We take the first `*_county*.csv` (or any `.csv` in that folder if needed).

In [ ]:
def load_nhgis_county_csv(zip_path: Path) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path, "r") as zf:
        names = zf.namelist()
        csv_candidates = [
            n for n in names
            if n.lower().endswith(".csv") and "_csv/" in n.replace("\\", "/")
        ]
        county_csv = [n for n in csv_candidates if "county" in n.lower()]
        chosen = (county_csv or csv_candidates)[0]
        with zf.open(chosen) as f:
            # csv_header: row 0 = names, row 1 = descriptions (skip row 1)
            df = pd.read_csv(f, header=0, skiprows=[1], low_memory=False)
    return df


raw = load_nhgis_county_csv(ZIP_PATH)
raw.head()

## Wide table: identifier columns + census year columns

NHGIS names data columns with a table/series prefix and a 4-digit year suffix (e.g. `A00AA2020`). We rename those columns to the census year string. Identifier columns are those that do not match that pattern.

In [ ]:
YEAR_SUFFIX = re.compile(r"^(?P<prefix>.+?)(?P<year>\d{4})$")


def split_id_and_year_columns(columns):
    id_cols = []
    year_map = {}
    for c in columns:
        s = str(c).strip()
        m = YEAR_SUFFIX.match(s)
        if m and 1790 <= int(m.group("year")) <= 2100:
            year_map[s] = m.group("year")
        else:
            id_cols.append(s)
    return id_cols, year_map


id_cols, year_map = split_id_and_year_columns(raw.columns)
wide = raw.rename(columns=year_map)
year_cols = sorted(year_map.values(), key=int)

# Keep rows that have 2020 data = proxy for "current" county set in NHGIS nominal file
if "2020" in wide.columns:
    mask = wide["2020"].notna() & (wide["2020"].astype(str).str.strip() != "")
    wide_current = wide.loc[mask].copy()
else:
    wide_current = wide.copy()

# Numeric population columns
for y in year_cols:
    wide_current[y] = pd.to_numeric(wide_current[y], errors="coerce")

out = wide_current[id_cols + year_cols]
out.to_csv(CSV_OUT, index=False)
print("Wrote", CSV_OUT, "shape", out.shape)

## Graphical summary: U.S. total population by census year

Sum population across counties for each census column (ignoring NaNs).

In [ ]:
totals = out[year_cols].sum(skipna=True)
years_int = [int(y) for y in totals.index]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(years_int, totals.values, marker="o", markersize=3)
ax.set_title("U.S. total population (sum of NHGIS A00 county cells)")
ax.set_xlabel("Census year")
ax.set_ylabel("Population")
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig_path = DATA_DIR / "us_total_population_timeseries.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print("Saved", fig_path)